In [1]:
import pandas as pd
import glob
import numpy as np
from evcouplings.compare import DistanceMap, ClassicPDB
from copy import deepcopy
from scipy.stats import ks_2samp
import os
import matplotlib.pyplot as plt
import seaborn as sns

from evcouplings.visualize.pymol import (
    pymol_pair_lines, pymol_mapping
)

import warnings
warnings.filterwarnings("ignore")

In [2]:
!mkdir summary_data

### step 1: determine if any of the hits have at least 90% coverage overall - will later filter to see if these are coming from a single hit
d = pd.read_csv("../structure_search/output_structural_data_v3_2024.csv", index_col=0)
d = d.dropna(subset=["uid", "bitscore"])
d = d.fillna(0)

n_proteins = len(d.uid.unique())

# merge with correct lengths
proteome = pd.read_csv("../structure_search/H37Rv_proteome.csv")
print("total number of proteins in proteome", len(proteome))
l = proteome[["uid", "Length"]]
d = d.merge(l, on="uid", how="right")

print(f"hits for a total of {len(d.uid.unique())} proteins")
d.to_csv("summary_data/00.structure_search_hits.csv")

# Partition the datasets. Remember that each uid was searched for 3 different bitscores, 
# so make sure to account for uid redundancy
structure_hits_possible_90percent = d.query("percent_covered >= .9")
uids_possible_90percent = set(structure_hits_possible_90percent.uid)

structure_hits_less_than_90percent = d.query("percent_covered < 0.9 and num_hits > 0 and run_successful and not uid in @uids_possible_90percent")

all_structure_uids = uids_possible_90percent.union(set(structure_hits_less_than_90percent.uid))

no_hits = d.query("num_hits==0 and run_successful and not uid in @all_structure_uids")

print("Proteins with no structural hits", len(no_hits.uid.unique()))
print("proteins with a putative 90% hit", len(structure_hits_possible_90percent.uid.unique()))
print("Proteins with no 90% hits", len(structure_hits_less_than_90percent.uid.unique()))


mkdir: cannot create directory ‘summary_data’: File exists
total number of proteins in proteome 3993
hits for a total of 3993 proteins
Proteins with no structural hits 1009
proteins with a putative 90% hit 1571
Proteins with no 90% hits 1409


In [3]:
structure_hits_possible_90percent.query("uid=='RECA_MYCTU'")

,uid,Length_x,run_successful,num_hits,num_hits_unfiltered,percent_covered,bitscore,Length_y


In [4]:
def select_best_distmap(df, length):
    
    for idx, row in df.iterrows():
        uid = row.uid
        bit = row.bitscore
        mapping_index = row.mapping_index
        
        dm_file = f"../structure_search/output/{uid}_b{bit}/compare/aux/{uid}_b{bit}_distmap_intra_{mapping_index}"
        
        if not os.path.isfile(dm_file+".npy"):
            continue
            
        dm = DistanceMap.from_file(dm_file)
        
        coverage = len(dm.residues_i) 
        if coverage >= 0.9*(length) and coverage <= length:
            #print("coverage", coverage, "length", length, dm_file)
            return idx, coverage
            
    return None, None

#### Determine if the coverage is coming from a single hit or multiple hits
#### If from a single hit, select the representative structure to use
#### Representative structures are taken from proteins with the same uniprot ID, if available, else take best hit

hit_information = []
uids_with_less_than_90percent = []

files_not_found = []
problem_uids = []

# Iterate through uids
for uid in structure_hits_possible_90percent.uid.unique():
    
    length = list(structure_hits_possible_90percent.query("uid==@uid").Length_x)[0]
    
    all_hits = pd.DataFrame()
    
    # Read in the structure hits for each bitscore and concatenate
    for bit in ["0.1", "0.2", "0.3", "1.2"]:
           # Read in the appropriate file
        try:
            s_hits = pd.read_csv(
                f"../structure_search/output/{uid}_b{bit}/compare/{uid}_b{bit}_structure_hits.csv"
            )
            s_hits["uid"] = uid
            s_hits["bitscore"] = bit
        except FileNotFoundError:
            files_not_found.append(uid)
            continue
        
        all_hits = pd.concat([all_hits, s_hits])
    #print(all_hits)
     
    # Take only hits with coverage > 90%
    all_hits["percent_coverage"] = all_hits.overlap / length
    all_hits = all_hits.reset_index()    
    top_hits = all_hits.query("percent_coverage >= .90")
    
    # guarantee that hits are sorted
    top_hits = top_hits.sort_values("e_value")
    #print(top_hits)
    
    if len(top_hits) ==0:
        uids_with_less_than_90percent.append(uid)
        continue
    
    # Take the hit with the highest similarity - if from MYCTU take that, else highest similarity
    self_hits = top_hits.query("uniprot_id==@uid")
    
    idx_to_keep = None
    
    if len(self_hits) > 0:
        idx_to_keep, coverage = select_best_distmap(self_hits, length)
        #print(idx_to_keep)
        
    if idx_to_keep is None:
        idx_to_keep, coverage = select_best_distmap(top_hits, length)
        
    if idx_to_keep is not None:
        top_hits.loc[idx_to_keep, "updated_coverage"] = coverage / length
        
        hit_information.append(top_hits.loc[idx_to_keep,:])
        
    elif idx_to_keep is None:
        problem_uids.append(uid)
        
hit_information = pd.DataFrame(hit_information)

print("Didn't find an individual structure hit with 90% coverage for", len(uids_with_less_than_90percent))
hit_information.to_csv("summary_data/01B.structure_search_hits_0p90_coverage.csv")

hit_information = pd.read_csv("summary_data/01B.structure_search_hits_0p90_coverage.csv", index_col=0)
print(len(hit_information), len(files_not_found), len(problem_uids))

Didn't find a single structure hit with 90% coverage for 0
1371 1704 200


In [5]:
uids_with_less_than_90percent += problem_uids

In [6]:
# ## Partition the dataset into alphafold and structure hits
# no_hits = d.query("not uid in @hit_information.uid")

no_hits = no_hits.drop_duplicates(["uid"])
print(len(no_hits))
no_hits.to_csv("summary_data/01A.structure_search_hits_no_hits.csv")

structure_hits_no_90percent = structure_hits_possible_90percent.query("uid in @uids_with_less_than_90percent")
structure_hits_less_than_90percent = pd.concat([structure_hits_no_90percent, structure_hits_less_than_90percent])
structure_hits_less_than_90percent = structure_hits_less_than_90percent.drop_duplicates(["uid"])
print(len(structure_hits_less_than_90percent))
structure_hits_less_than_90percent.to_csv("summary_data/01C.structure_search_hits_less_than_90_percent.csv")

1009
1609


## Generate distancemaps for alphafold predictions

In [10]:
hit_information = pd.read_csv("summary_data/01A.structure_search_hits_no_hits.csv", index_col=0)
hit_information2 = pd.read_csv("summary_data/01C.structure_search_hits_less_than_90_percent.csv", index_col=0)
hit_information = pd.concat([hit_information, hit_information2])

id_mapping = pd.read_csv("../GeO_2021_09_15//summary_data/h37rv_gene_ids_to_uniprot_ids.csv", index_col=0)
hit_information = hit_information.merge(id_mapping[["Entry name", "Entry", "rv"]], how="left", left_on="uid", right_on="Entry name")

alphafold_filtering_information = []

! mkdir ../alphafold/UP000001584_83332_MYCTU_v4_b70_filtered
! mkdir alphafold_distmaps

skip = False

for idx, row in hit_information.iterrows():

    uid = row.uid
    entry = row.Entry
    
    if skip and os.path.isfile(f"alphafold_distmaps/{entry}.csv"):
        print(uid, "already completed")
        continue
        
    print("running", uid)
    
    try:
        structure = ClassicPDB.from_file(f"../alphafold/UP000001584_83332_MYCTU_v4/AF-{entry}-F1-model_v4.pdb")
    except: 
        print("no file for ", entry)
        continue

    chain = structure.get_chain("A")

    current_pdb_low_b_factor = list(chain.coords.query("b_factor < 70").residue_index)
    num_residues_initial = len(chain.coords.residue_index.unique())
    
    residues_to_keep = chain.coords.query("b_factor >= 70").residue_index
    print(num_residues_initial, len(residues_to_keep))
    chain2 = chain.filter_positions(residues_to_keep)
    dm = DistanceMap.from_coords(chain2)

    dm.to_file(f"alphafold_distmaps/{entry}")

    chain2.to_file(open(f"../alphafold/UP000001584_83332_MYCTU_v4_b70_filtered/AF-{entry}-F1-model_v4_beta70filtered.pdb", "w"))
    
    alphafold_filtering_information.append([entry, uid, num_residues_initial, len(residues_to_keep.unique())])

alphafold_filtering_information = pd.DataFrame(alphafold_filtering_information, columns=["entry", "uid", "residues_initial", "residues_filtered"])

alphafold_filtering_information.to_csv("summary_data/alphafold_filtering_information.csv")


mkdir: cannot create directory ‘../alphafold/UP000001584_83332_MYCTU_v4_b70_filtered’: File exists
mkdir: cannot create directory ‘alphafold_distmaps’: File exists
running Y876_MYCTU
559 2280
running Y2076_MYCTU
83 50
running Y037_MYCTU
441 2945
running Y1995_MYCTU
255 1541
running Y2012_MYCTU
164 1066


KeyboardInterrupt: 

In [11]:
alphafold_filtered = pd.read_csv("summary_data/alphafold_filtering_information.csv")
print(len(alphafold_filtered))
alphafold_filtered.query("residues_filtered > 0")

2614


,Unnamed: 0,entry,uid,residues_initial,residues_filtered
0,0,P9WKR5,Y876_MYCTU,559,315
1,1,P9WLL3,Y2076_MYCTU,83,6
2,2,P9WJY1,Y037_MYCTU,441,402
3,3,P9WLP3,Y1995_MYCTU,255,192
4,4,P9WLM3,Y2012_MYCTU,164,136
...,...,...,...,...,...
2609,2609,I6Y9H2,I6Y9H2_MYCTU,185,158
2610,2610,L0TB61,L0TB61_MYCTU,324,271
2611,2611,L7N662,L7N662_MYCTU,485,456
2612,2612,O53340,O53340_MYCTU,340,339


## Filter the EBR and mappability regions from all distance maps

In [12]:
!mkdir input
!cp ../2023_02_23_benchmark/input/AF7_H37Rv_FeatureLevelAnalysis.EBR_And_Pmap.tsv input
!cp ../2023_02_23_benchmark/input/EBR.csv input
!cp ../2023_02_23_benchmark/input/S7-seq-attributes.csv input
!cp ../GeO_2021_09_21/summary_data/h37rv_gene_ids_to_uniprot_ids.csv input

In [13]:
## Get a list of proteins with poor EBR and/or mappability

mappability = pd.read_csv("input/AF7_H37Rv_FeatureLevelAnalysis.EBR_And_Pmap.tsv", sep="\t")
mappability = mappability.query('Feature == "CDS"')

print("all proteins in Marin et al", len(mappability))
print("after filtering", len(mappability.query("Mean_EBR_36CI >= .9 and Mean_PileupMapScore_K50_E4 >= .9")))

proteins_to_keep = mappability.query("Mean_EBR_36CI >= .9 and Mean_PileupMapScore_K50_E4 >= .9")


all proteins in Marin et al 4031
after filtering 3844


In [14]:
## Combined file of structure hits and alphafold hits
## filter for EBR, mappability

In [15]:
alphafold_hits = pd.read_csv("summary_data/01A.structure_search_hits_no_hits.csv", index_col=0)
alphafold_hits["set"] = "setA"

structure_hits = pd.read_csv("summary_data/01B.structure_search_hits_0p90_coverage.csv", index_col=0)
structure_hits["set"] = "setB"

structure_hits2 = pd.read_csv("summary_data/01C.structure_search_hits_less_than_90_percent.csv", index_col=0)
structure_hits2["set"] = "setA"

all_hits = pd.concat([alphafold_hits, structure_hits, structure_hits2])

# Get the H37rv locus tag
id_mapping = pd.read_csv("input/h37rv_gene_ids_to_uniprot_ids.csv", index_col=0)
all_hits = all_hits.merge(id_mapping[["Entry name", "Entry", "rv"]], how='left', left_on="uid", right_on="Entry name")

# Filtering
all_hits = all_hits.query("rv in @proteins_to_keep.H37rv_GeneID")
all_hits.query("uid=='EMBB_MYCTU'")

,uid,Length_x,run_successful,num_hits,num_hits_unfiltered,percent_covered,bitscore,Length_y,set,index,...,uniprot_end,uniprot_id_,overlap,mapping_index,grouped_segments,percent_coverage,updated_coverage,Entry name,Entry,rv
1802,EMBB_MYCTU,NaN,NaN,NaN,NaN,NaN,0.1,NaN,setB,0.0,...,1098.0,EMBB_MYCTU,1054.0,1.0,True,0.959927,0.959927,EMBB_MYCTU,P9WNL7,Rv3795


In [16]:
# Prepare a list of sites in each gene that can be mapped
genome_position_to_gene = pd.read_csv("../../databases/genome_position_to_gene_with_codons.csv")
genome_position_to_gene = genome_position_to_gene.drop("ebr", axis=1)

# Ensure that we've accurately filtered positions from Marin et al
ebr = pd.read_csv(
    "input/EBR.csv", index_col=0
)
genome_position_to_gene = genome_position_to_gene.merge(ebr, left_on="i", right_on="position", how="left")

# Accurately filter the positions from Modlin et al
modlin = pd.read_csv("input/S7-seq-attributes.csv")[["rv_position", "blindspot"]]
blindspots = modlin.query("blindspot==1")
blindspot_list = list(blindspots.rv_position)
genome_position_to_gene = genome_position_to_gene.merge(modlin, left_on="i", right_on="rv_position", how="left")

genome_position_to_gene.to_csv("input/gene_positions_included.csv")


In [17]:
# Remove proteins with no alphafold residues
alphafold_filtering = pd.read_csv("summary_data/alphafold_filtering_information.csv", index_col=0)
no_coverage = alphafold_filtering.query("residues_filtered==0").uid

all_hits = all_hits.query("not uid in @no_coverage")

## Iterate through all distance maps, create filtered version, save to the same directory

In [41]:
!mkdir filtered_distmaps

mkdir: cannot create directory ‘filtered_distmaps’: File exists


In [18]:
# Quick patch - the alphafold structures don't have a chain_index in their output distancemap.csv files
# So, let's read them and set the chain id to the same as the coord id

for idx, row in all_hits.iterrows():
    entry = row["Entry"]

    if row.set=='setA': #alphafold distancemap
        try:
            data = pd.read_csv(f"alphafold_distmaps/{entry}.csv", index_col=0)
            data.chain_index = data.coord_id
            data.to_csv(f"alphafold_distmaps/{entry}.csv")
        except:
            continue

In [19]:
def filter_distmatrix(dm, residues_to_filter):
    print("length of dist matrix before filtering", len(dm.residues_i))
    dm.residues_i.reset_index(drop=True, inplace=True)
    dm.residues_i = dm.residues_i.query("id not in @residues_to_filter")
    
    print("length of dist matrix after filtering", len(dm.residues_i))
    keep_indices = np.array([x for x in dm.residues_i.index])
    dm.dist_matrix = dm.dist_matrix[keep_indices,:][:, keep_indices]
    
    return dm

In [21]:
! mkdir output
! mkdir filtered_distmaps
excluded_ids = []
skip=True

all_hits["filtered_length"] = np.nan

for idx, row in all_hits.iterrows():
    uid = row.uid
    
#     if not uid=='EMBB_MYCTU':
#         continue
    
    rv = row.rv
    entry = row["Entry"]
    #print(entry)
    os.system(f"mkdir output/{uid}")   
    
    if skip and os.path.isfile(f"filtered_distmaps/{entry}.csv"):
        continue
        
    if row.set=='setA': #alphafold distancemap
        try:
            dm = DistanceMap.from_file(f"alphafold_distmaps/{entry}")
        except:
            print("no alphafold distance map", entry)
            excluded_ids.append(uid)
            continue
            
    elif row.set=='setB': #structure distancemap
    
        bit = row.bitscore
        mapping_index = int(row.mapping_index)
        prefix = f"{uid}_b{bit}_m{mapping_index}"

        # Read in the distance map
        try:
            dm = DistanceMap.from_file(f"../structure_search/output/{uid}_b{bit}/compare/aux/{uid}_b{bit}_distmap_intra_{mapping_index}")
        except:
            print("no distance map", f"../structure_search/output/{uid}_b{bit}/compare/aux/{uid}_b{bit}_distmap_intra_{mapping_index}")
            continue
    
    print(row.set, dm.dist_matrix.shape)
    
    current_locus_mappability = genome_position_to_gene.query("locus==@rv")
    current_locus_unmappable = current_locus_mappability.query("blindspot==1 or EBR < 0.9").codon.astype(int).astype(str).unique()
    df_unmappable = pd.DataFrame(data=current_locus_unmappable, columns=["i"])
    df_unmappable.to_csv(f"output/{uid}/unmappable_residues.csv")  
    
    if len(current_locus_unmappable) == 0:
        pass
    elif len(current_locus_unmappable) < len(dm.residues_i):
        print(f"Filtering {len(current_locus_unmappable)} from distmatrix with shape {dm.dist_matrix.shape}")
        dm = filter_distmatrix(dm, current_locus_unmappable)
        print("Result has shape", dm.dist_matrix.shape)
    else:
        print("all the positions are unmappable", uid)
        excluded_ids.append(uid)
        continue
        
    all_hits.loc[idx, "filtered_length"] = len(dm.residues_i)
        
    
    dm.to_file(f"filtered_distmaps/{entry}")
    
    #break

mkdir: cannot create directory ‘output’: File exists


mkdir: cannot create directory ‘output/Y876_MYCTU’: File exists


setA (315, 315)
setA (6, 6)
setA (402, 402)
setA (192, 192)
Filtering 3 from distmatrix with shape (192, 192)
length of dist matrix before filtering 192
length of dist matrix after filtering 189
Result has shape (189, 189)
setA (136, 136)
setA (248, 248)
setA (372, 372)
setA (256, 256)
setA (263, 263)
Filtering 2 from distmatrix with shape (263, 263)
length of dist matrix before filtering 263
length of dist matrix after filtering 261
Result has shape (261, 261)
setA (78, 78)
setA (150, 150)
setA (326, 326)
setA (88, 88)
setA (127, 127)
setA (253, 253)
setA (138, 138)
setA (55, 55)
setA (111, 111)
setA (56, 56)
setA (341, 341)
setA (328, 328)
setA (112, 112)
setA (99, 99)
setA (918, 918)
setA (181, 181)
setA (325, 325)
setA (352, 352)
setA (508, 508)
setA (60, 60)
setA (232, 232)
setA (229, 229)
setA (422, 422)
Filtering 88 from distmatrix with shape (422, 422)
length of dist matrix before filtering 422
length of dist matrix after filtering 354
Result has shape (354, 354)
setA (532, 532

In [22]:
## Abbreviated version - just gets the length of hte already filtered distmaps, no filtering performed
all_hits["filtered_length"] = np.nan
excluded_ids = []

for idx, row in all_hits.iterrows():
    uid = row.uid
    entry = row.Entry

    
    if os.path.isfile(f"filtered_distmaps/{entry}.csv"):
        dm = DistanceMap.from_file(f"filtered_distmaps/{entry}")
        all_hits.loc[idx, "filtered_length"] = len(dm.residues_i)
        
    else:
        excluded_ids.append(uid)


## Clean up the protein hit table

In [23]:
all_hits = all_hits.query("not uid in @excluded_ids")
all_hits[["uid", "Entry", "rv",  "set", 'pdb_id', 'pdb_chain', 'filtered_length']].to_csv("summary_data/01.structure_hits.csv")

In [24]:
all_hits = pd.read_csv("summary_data/01.structure_hits.csv", index_col=0)
print(len(all_hits))
print(len(all_hits.query("not filtered_length < 30")))
all_hits.query("not filtered_length < 30").to_csv("summary_data/01.structure_hits_filtered.csv")

3758
3687


In [25]:
all_hits = pd.read_csv("summary_data/01.structure_hits_filtered.csv")

In [26]:
len(all_hits.query("set=='setA'")), len(all_hits.query("set=='setB'"))

(2337, 1350)

In [27]:
2337+1350

3687